# Experiment Comparison / Error Analysis

`runs/summary.csv` に記録された複数Runの結果を比較し、選択したRunの誤判定を確認します。

改善サイクル: 誤判定を確認 → 原因仮説 → Config変更（前処理・Dataset・学習条件・Threshold） → `01` / `02` のNotebookで再実行（Thresholdのみなら `reevaluate`） → ここで比較。

指標はすべて各Runの **test**（Threshold決定に使っていないデータ）での値です。`threshold_mode=manual` のRunは、Thresholdをtestの結果を見て決めた場合は独立した評価ではありません。

## 1. Run Summary

In [ ]:
%matplotlib inline
from pathlib import Path

from IPython.display import display

from anomaly_detection.experiments.run_manager import load_summary
from anomaly_detection.utils.config import find_project_root
from anomaly_detection.utils.logging import setup_logging

setup_logging()
RUN_ROOT = find_project_root(Path.cwd()) / "runs"
summary = load_summary(RUN_ROOT)
HAS_RUNS = not summary.empty
if not HAS_RUNS:
    print(f"Runがありません: {RUN_ROOT / 'summary.csv'}")
    print("01_phase1_mvtec_vial.ipynb または 02_phase2_walnut.ipynb を先に実行してください。")
else:
    print(f"{len(summary)} Runs")

## 2. Metrics Comparison Table

`PHASE` で対象を絞り込み、`LAST_N` で直近のRunを表示します。

In [ ]:
PHASE = "phase1"  # 'phase1' / 'phase2' / None（すべて）
LAST_N = 10

COLUMNS = [
    "run_id",
    "phase",
    "preprocessing_summary",
    "threshold_mode",
    "threshold",
    "recall",
    "precision",
    "f1",
    "accuracy",
    "auroc",
    "aupr",
    "pixel_auroc",
    "pixel_aupro",
    "fn",
    "fp",
    "num_images",
    "retrained",
    "parent_run_id",
]
if HAS_RUNS:
    selected = summary if PHASE is None else summary[summary["phase"] == PHASE]
    selected = selected.tail(LAST_N)
    display(selected[COLUMNS])

## 3. Recall / Precision / F1 / Threshold 比較

外観検査では見逃し（FN）を重視するため、まずRecallを確認します。ThresholdはPatchCoreの生スコア単位のため、学習条件が異なるRun間では値の大小を直接比較できません。

In [ ]:
from anomaly_detection.visualization.run_comparison import plot_run_comparison

if HAS_RUNS and not selected.empty:
    display(plot_run_comparison(selected, metrics=("recall", "precision", "f1", "threshold")))

## 4. False Positive / False Negative

`RUN_ID` のRunについて、testの誤判定を表示します（未指定なら上の表の最新Run）。Runの `predictions.csv` と `anomaly_maps.npz` を読み込むため、再推論は行いません。

In [ ]:
from anomaly_detection.evaluation.error_analysis import (
    get_false_negatives,
    get_false_positives,
    plot_errors,
)
from anomaly_detection.experiments.run_manager import load_run
from anomaly_detection.preprocessing.pipeline import ImagePreprocessor
from anomaly_detection.visualization.anomaly_visualizer import HeatmapScale

RUN_ID = None  # 例: '20260926_225405_phase1'
MAX_ITEMS = 6

run = None
if HAS_RUNS and not selected.empty:
    run = load_run(RUN_ROOT / (RUN_ID or selected["run_id"].iloc[-1]))
    preprocessor = ImagePreprocessor(run.config.preprocessing)
    scale = HeatmapScale(**run.metadata["heatmap_scale"])
    false_negatives = get_false_negatives(run.predictions)
    false_positives = get_false_positives(run.predictions)
    print("Run:", run.run.run_id, "| Threshold:", run.metadata["threshold"])
    print(f"False Negative（見逃し）: {len(false_negatives)} 件")
    print(f"False Positive（過検知）: {len(false_positives)} 件")

### False Negative（異常だがGOODと判定）

In [ ]:
COLUMNS_ERR = ["image_path", "ground_truth", "defect_type", "prediction", "anomaly_score"]
if run is not None:
    display(false_negatives[COLUMNS_ERR])
    errors = false_negatives.assign(error_type="FN")
    figures = plot_errors(errors, run.anomaly_maps, preprocessor, scale, max_items=MAX_ITEMS)
    for figure in figures:
        display(figure)

### False Positive（正常だがNGと判定）

In [ ]:
if run is not None:
    display(false_positives[COLUMNS_ERR])
    errors = false_positives.assign(error_type="FP")
    figures = plot_errors(errors, run.anomaly_maps, preprocessor, scale, max_items=MAX_ITEMS)
    for figure in figures:
        display(figure)